In [ ]:
import numpy as np                                                                         
from phasic import Graph, StateIndexer, Property, with_ipv

from itertools import combinations_with_replacement

N_SAMPLES = 5
EPOCH_BOUNDARY = 0.5
TRUE_THETA1 = 2.0
TRUE_THETA2 = 0.5
MUTATION_RATE = 0.01
REWARD_LIMIT = 5      # cap on mutations per locus the joint graph tracks

# Indexer over the lineage state — one Property: descendants count per block.
indexer = StateIndexer(
    lineages=[Property('descendants', min_value=1, max_value=N_SAMPLES)],
)
ipv = [0] * indexer.state_length
ipv[indexer.lineages.props_to_index(descendants=1)] = N_SAMPLES  # start with N_SAMPLES singletons


@with_ipv(ipv)
def coal_callback(state, indexer=None):
    transitions = []
    for i, j in combinations_with_replacement(range(indexer.lineages.state_length), 2):
        same = int(i == j)
        if same and state[i] < 2:
            continue
        if not same and (state[i] < 1 or state[j] < 1):
            continue
        new = state.copy()
        new[i] -= 1
        new[j] -= 1
        new[min(i + j + 1, state.size - 1)] += 1
        pair_count = state[i] * (state[j] - same) / (1 + same)
        transitions.append([new, [pair_count]])
    return transitions


def sample_path_with_continuous_time(jg, theta_proposal, target_vertex, n=1, rng=None):      
    """
    Sample paths through a (discrete-time) joint_prob_graph conditioned on                   
    ending at `target_vertex`, and map each step's discrete entry time back                  
    to continuous time by resampling sojourns from Exp(R_v_raw), where                       
    R_v_raw is the un-normalized exit rate from raw edge coefficients.                       
                                                                                            
    Parameters                                                                               
    ----------                                                                               
    jg : Graph                                                                             
        A joint_prob_graph (is_discrete=True, was_dph=True). Its edge
        coefficients must already encode the proposal CTMC.                                  
    theta_proposal : array-like, shape (n_params,)                                           
        Proposal parameter vector. Used both to set the graph weights for                    
        h-guided sampling AND to compute raw exit rates R_v_raw.                             
    target_vertex : int                                                                      
        Index of the t-vertex (terminal observation) to condition on.                        
    n : int                                                                                  
        Number of paths to sample.                                                         
    rng : np.random.Generator, optional                                                      
                                                                                            
    Returns                                                                                  
    -------                                                                                
    list of list of (state_tuple, entry_time_float)
        One list per path. Each element is (vertex_state, continuous entry time).            
        First element has entry_time=0.0 (starting vertex).                                  
        Last element is the absorbing target vertex.                                         
    """                                                                                      
    rng = rng or np.random.default_rng()                                                     
    theta_proposal = np.asarray(theta_proposal, dtype=float)                                 
                                                                                            
    # 1. Set weights so the C sampler's h-guidance reflects the proposal CTMC.               
    jg.update_weights(theta_proposal.tolist())                                               
                                                                                            
    # 2. Precompute raw (un-normalized) exit rate R_v_raw for every vertex,                  
    #    using the original edge coefficients (these are NOT touched by                      
    #    update_weights's row-normalization on a was_dph graph).                             
    n_params = jg.param_length()                                                             
    vertices = jg.vertices()                                                                 
    R_raw = np.zeros(jg.vertices_length(), dtype=float)                                      
    for v in vertices:                                                                       
        if v.edges_length() == 0:                                                            
            continue                                                                         
        edges = v.parameterized_edges()                                                    
        if not edges:                                                                        
            continue                                                                         
        coeffs = np.array([list(e.edge_state(n_params)) for e in edges])
        if coeffs.ndim == 1:                                                                 
            coeffs = coeffs.reshape(1, -1)                                                 
        if coeffs.shape[1] < n_params:                                                       
            coeffs = np.pad(coeffs, ((0, 0), (0, n_params - coeffs.shape[1])))             
        edge_rates = coeffs @ theta_proposal      # shape (n_edges,)                         
        R_raw[v.index()] = edge_rates.sum()                                                  
                                                                                            
    # 3. Sample n paths. The C sampler returns Exp(1) sojourns on a was_dph                  
    #    graph, so we ignore its entry_times and resample from Exp(R_v_raw).               
    paths = jg.sample_path_conditioned([int(target_vertex)], n=n)                            
    if n == 1:                                                                               
        paths = [paths]                                                                      
                                                                                            
    out = []                                                                                 
    for path in paths:                                                                     
        indices = np.asarray(path['vertex_indices'], dtype=int)
        states = [tuple(int(x) for x in vertices[int(vi)].state()) for vi in indices]        
        n_steps = len(indices)                                                               
                                                                                            
        cont_times = np.zeros(n_steps, dtype=float)                                          
        # cont_times[0] = 0.0 (start). For step k we draw a sojourn at vertex              
        # indices[k] and add it to cont_times[k] to get cont_times[k+1].                     
        for k in range(n_steps - 1):                                                         
            v_idx = int(indices[k])                                                          
            r = R_raw[v_idx]                                                                 
            if r > 0.0:                                                                      
                cont_times[k + 1] = cont_times[k] + rng.exponential(1.0 / r)
            else:                                                                            
                # Absorbing or unreachable; no further continuous time accrues.            
                cont_times[k + 1] = cont_times[k]                                            
        out.append(list(zip(states, cont_times.tolist())))                                 
                                                                                            
    return out                                                                             

graph = Graph(coal_callback, indexer=indexer)

# Usage:                                                                                     

# Build joint-prob graph (already discrete + was_dph).
jg = graph.joint_prob_graph(indexer, mutation_rate=MUTATION_RATE, reward_limit=REWARD_LIMIT)                       


jg.update_weights([TRUE_THETA1, MUTATION_RATE])                                              
                                                                                            
# 2. The joint probability table has one row per terminal (t-)vertex.                        
#    Its index is the vertex index; the leading columns are the feature counts.
jpt = jg.joint_prob_table()                                                                  
# Example layout:                                                                            
#                 feat0  feat1  feat2     prob                                               
# vertex_idx                                                                                 
#       12          0      1      0     0.213
#       17          1      0      0     0.456                                                
#       ...     
                                                                                            
# 3. Map a feature-count tuple -> vertex index.                                              
obs2idx = jpt.groupby(jpt.columns[:-1].to_list()).groups
# obs2idx[(0, 1, 0)] -> Index([12])                                                          
obs2idx

{(0, 0, 0, 0, 0): [20], (0, 0, 0, 1, 0): [49], (0, 0, 0, 2, 0): [90], (0, 0, 0, 3, 0): [148], (0, 0, 1, 0, 0): [44], (0, 0, 1, 1, 0): [116], (0, 0, 1, 2, 0): [186], (0, 0, 1, 3, 0): [271], (0, 0, 2, 0, 0): [83], (0, 0, 2, 1, 0): [188], (0, 0, 2, 2, 0): [272], (0, 0, 2, 3, 0): [363], (0, 0, 3, 0, 0): [141], (0, 0, 3, 1, 0): [274], (0, 0, 3, 2, 0): [364], (0, 0, 3, 3, 0): [451], (0, 1, 0, 0, 0): [41], (0, 1, 0, 1, 0): [106], (0, 1, 0, 2, 0): [172], (0, 1, 0, 3, 0): [254], (0, 1, 1, 0, 0): [81], (0, 1, 1, 1, 0): [204], (0, 1, 1, 2, 0): [295], (0, 1, 1, 3, 0): [393], (0, 1, 2, 0, 0): [139], (0, 1, 2, 1, 0): [297], (0, 1, 2, 2, 0): [394], (0, 1, 2, 3, 0): [487], (0, 1, 3, 0, 0): [217], (0, 1, 3, 1, 0): [396], (0, 1, 3, 2, 0): [488], (0, 1, 3, 3, 0): [566], (0, 2, 0, 0, 0): [78], (0, 2, 0, 1, 0): [174], (0, 2, 0, 2, 0): [255], (0, 2, 0, 3, 0): [346], (0, 2, 1, 0, 0): [137], (0, 2, 1, 1, 0): [300], (0, 2, 1, 2, 0): [398], (0, 2, 1, 3, 0): [490], (0, 2, 2, 0, 0): [215], (0, 2, 2, 1, 0): [400],

In [6]:

obs = (0, 0, 2, 3, 0)

target_v = int(obs2idx[tuple(obs)][0])                                            

sample_path_with_continuous_time(                                                
        jg, theta_proposal=[TRUE_THETA1, MUTATION_RATE],                                                   
        target_vertex=target_v, n=200,                                                       
    )   

[[((0, 0, 0, 0, 0, 0, 0, 0, 0, 0), 0.0),
  ((5, 0, 0, 0, 0, 0, 0, 0, 0, 0), 0.16754985885582557),
  ((3, 1, 0, 0, 0, 0, 0, 0, 0, 0), 0.3803084146511154),
  ((2, 0, 1, 0, 0, 0, 0, 0, 0, 0), 0.38468904554375744),
  ((2, 0, 1, 0, 0, 0, 0, 1, 0, 0), 0.3974695077219354),
  ((2, 0, 1, 0, 0, 0, 0, 2, 0, 0), 0.8185981636673366),
  ((1, 0, 0, 1, 0, 0, 0, 2, 0, 0), 1.0094343265358368),
  ((1, 0, 0, 1, 0, 0, 0, 2, 1, 0), 1.0397046069544431),
  ((1, 0, 0, 1, 0, 0, 0, 2, 2, 0), 1.228789870194644),
  ((1, 0, 0, 1, 0, 0, 0, 2, 3, 0), 3.1084105598890854),
  ((0, 0, 0, 0, 1, 0, 0, 2, 3, 0), 3.2746427676590697)],
 [((0, 0, 0, 0, 0, 0, 0, 0, 0, 0), 0.0),
  ((5, 0, 0, 0, 0, 0, 0, 0, 0, 0), 0.04689156287487077),
  ((3, 1, 0, 0, 0, 0, 0, 0, 0, 0), 0.05095099100988705),
  ((2, 0, 1, 0, 0, 0, 0, 0, 0, 0), 0.09145277397585225),
  ((2, 0, 1, 0, 0, 0, 0, 1, 0, 0), 0.15402257276954137),
  ((2, 0, 1, 0, 0, 0, 0, 2, 0, 0), 0.19059604991633003),
  ((1, 0, 0, 1, 0, 0, 0, 2, 0, 0), 0.20170396797626233),
  ((1, 0, 0, 1

In [ ]:
#   For a list of observations:                                                                  
                
def observed_to_terminals(jg, observed_data, theta_proposal):                                
    jg.update_weights(np.asarray(theta_proposal).tolist())                                   
    jpt = jg.joint_prob_table()                                                              
    obs2idx = jpt.groupby(jpt.columns[:-1].to_list()).groups                                 
    targets = []                                                                             
    for obs in observed_data:
        idx = obs2idx[tuple(obs)]                                                            
        if len(idx) == 1:                                                                    
            targets.append(int(idx[0]))
        else:                                                                                
            # Tie: multiple terminal vertices match this observation.
            # Pick proportionally to their probability mass (same heuristic                  
            # as probability_matching).                                                      
            p = jpt.loc[idx, 'prob'].to_numpy()                                              
            p = p / p.sum()                                                                  
            targets.append(int(np.random.choice(idx, p=p)))
    return targets                                                                           



In [24]:
observed_data = [obs]

#So the full pipeline becomes:                                                                

n_path_samples = 5
jg = graph.joint_prob_graph(indexer, mutation_rate=MUTATION_RATE, reward_limit=REWARD_LIMIT)                       
targets = observed_to_terminals(jg, observed_data, [TRUE_THETA1, MUTATION_RATE])                           
                                                                                            
for obs, target_v in zip(observed_data, targets):                                            
    paths = sample_path_with_continuous_time(                                                
        jg, theta_proposal=[TRUE_THETA1, MUTATION_RATE],                                                   
        target_vertex=target_v, n=n_path_samples,                                                       
    )
    for path in paths:
        states, times = zip(*path)
        print(times)
        state_idx = np.searchsorted(times, EPOCH_BOUNDARY) - 1
        assert state_idx < len(times)
        assert times[state_idx] < EPOCH_BOUNDARY
        print(states[state_idx])
        vertex = graph.find_vertex((1, 0, 0, 1, 0, 0, 0, 2, 0, 0))
        print(vertex.index())
    # times = []
    # paths = [p for state, time in paths]
    # print(paths)                                                                                 
    # # paths[i] is [(state_tuple, continuous_time), ...]                                      
    


: 

In [ ]:

# Pick the t-vertex you want to condition on. For an SFS-style model this is                 
# the terminal vertex matching your observed locus.                                          
jpt = jg.joint_prob_table()                  # also sets weights internally                  
target_v = int(observed_terminal_vertex_idx) # one of jpt.index                              
                                                                                            
paths = sample_path_with_continuous_time(                                                    
    jg, theta_proposal=[TRUE_THETA1], target_vertex=target_v, n=200                                  
)                                                                                            
                                                                                            
for state, t in paths[0]:                                                                    
    print(t, state)


In [ ]:
                                                                                               
  Two things to watch:                                                                       

  1. update_weights must be called with the proposal theta before sampling, because the C-side 
  h-guidance uses the row-normalized weights derived from those coefficients. The snippet does
  this in step 1.                                                                              
  2. The continuous times are stochastic — a fresh resample per call. If you use these inside
  an importance-weight calculation, the same resample should be reused for both the            
  proposal-density and target-density terms (otherwise the ratio is biased). The snippet
  returns the resampled times, so just keep them around if you need them downstream.           
                                                                                             
  Want me to also add the matching importance-weight term (proposal vs. target CTMC density) so
   the same paths can drop into a BFFG-style estimator?